# Demostración del Pipeline de IA — Asistente de Pricing de Segunda Mano

Este notebook muestra el funcionamiento completo del módulo de IA del proyecto
**product_pricing_IA** sin necesidad de API key ni base de datos.  
Todos los pasos usan el modo **stub** para que puedas ejecutar el notebook
en local desde el primer momento.

---

## ¿Qué hace el módulo de IA?

1. **LLMClient** (GPT-4o vision): recibe hasta 9 fotos de un producto, las analiza y
   devuelve descripción, precio estimado, confianza y palabras clave de búsqueda.
2. **ExternalComparableClient**: busca comparables en internet usando las palabras clave
   del LLM y devuelve precios de mercado en tiempo real.
3. **PricingService**: combina ambas señales con una ponderación `LLM×0.6 + mercado×0.4`
   y genera la propuesta final con banda `[min, max]` y trazabilidad completa.

El operador revisa la propuesta en la interfaz de carrusel, puede corregirla, y esas
correcciones se retroalimentan automáticamente al prompt del siguiente análisis.

---

**Requisito RDA-002**: notebook paralelo con metodología, resultados y conclusiones.

## 1. Importar librerías y configurar el entorno

Usamos exclusivamente la librería estándar de Python y los módulos del propio proyecto.
No se necesita ninguna dependencia adicional para ejecutar el notebook en modo stub.

In [ ]:
import os
import sys
import random
import base64
import json
from pathlib import Path
from dataclasses import dataclass, field

# Aseguramos que la raíz del repositorio está en el path para importar módulos del proyecto
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Activar modo stub: no realiza llamadas reales a OpenAI
os.environ.setdefault("LLM_STUB", "true")
os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("DATABASE_URL", "sqlite:///./demo.db")

print("Entorno configurado:")
print(f"  LLM_STUB          = {os.environ['LLM_STUB']}")
print(f"  OPENAI_API_KEY    = {'<configurada>' if os.environ.get('OPENAI_API_KEY') else '<vacía — modo stub>'}")
print(f"  Repositorio raíz  = {REPO_ROOT}")

---
## 2. Carga y preprocesamiento de imágenes de producto

En producción, el usuario sube fotos vía `POST /api/v1/images/{product_id}`.
El sistema las almacena en disco y las convierte a base64 para enviarlas al LLM.

Aquí simulamos ese proceso con una imagen real del directorio `data/uploads/`
(o una imagen sintética si no hay uploads disponibles).

In [ ]:
def load_image_as_b64(path: Path) -> str:
    """Lee una imagen desde disco y la convierte a base64 (mismo formato que el backend)."""
    return base64.b64encode(path.read_bytes()).decode()


def create_synthetic_image_b64(label: str = "DEMO") -> str:
    """Genera una imagen PNG mínima 10x10 como placeholder cuando no hay uploads."""
    import struct, zlib

    def png_chunk(tag: bytes, data: bytes) -> bytes:
        chunk = tag + data
        return struct.pack(">I", len(data)) + chunk + struct.pack(">I", zlib.crc32(chunk) & 0xFFFFFFFF)

    header = b"\x89PNG\r\n\x1a\n"
    ihdr_data = struct.pack(">IIBBBBB", 10, 10, 8, 2, 0, 0, 0)
    ihdr = png_chunk(b"IHDR", ihdr_data)
    # 10 filas de 10 píxeles RGB azulado
    raw = b"".join(b"\x00" + bytes([0x4A, 0x7C, 0xC2] * 10) for _ in range(10))
    idat = png_chunk(b"IDAT", zlib.compress(raw))
    iend = png_chunk(b"IEND", b"")
    png_bytes = header + ihdr + idat + iend
    return base64.b64encode(png_bytes).decode()


# Buscar imágenes reales en data/uploads/
uploads_root = REPO_ROOT / "data" / "uploads"
image_files = list(uploads_root.rglob("*.jpg")) + list(uploads_root.rglob("*.jpeg")) + list(uploads_root.rglob("*.png"))

if image_files:
    # Tomamos hasta 3 imágenes reales
    selected = image_files[:3]
    photos = [
        {
            "filename": p.name,
            "content_base64": load_image_as_b64(p),
            "storage_uri": str(p),
        }
        for p in selected
    ]
    print(f"Imágenes reales cargadas: {len(photos)}")
    for ph in photos:
        size_kb = len(ph["content_base64"]) * 3 / 4 / 1024
        print(f"  - {ph['filename']}  ({size_kb:.1f} KB codificados en base64)")
else:
    # Fallback: imagen sintética
    photos = [
        {
            "filename": "demo_product.png",
            "content_base64": create_synthetic_image_b64("DEMO"),
            "storage_uri": "",
        }
    ]
    print("No se encontraron imágenes en data/uploads/ — usando imagen sintética de demostración.")
    print(f"  - {photos[0]['filename']}  ({len(photos[0]['content_base64']) * 3/4/1024:.1f} KB)")

---
## 3. Visualización de las imágenes de entrada

Mostramos las imágenes que se enviarían al LLM para que puedas verificar
que el preprocesamiento es correcto antes de la evaluación.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
    import io

    n = len(photos)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, photo in zip(axes, photos):
        raw = base64.b64decode(photo["content_base64"])
        img = mpimg.imread(io.BytesIO(raw), format="PNG" if photo["filename"].endswith(".png") else "JPEG")
        ax.imshow(img)
        ax.set_title(photo["filename"], fontsize=9)
        ax.axis("off")

    plt.suptitle("Imágenes de entrada al pipeline de IA", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"Mostrando {n} imagen(es) — resolución original, sin redimensionar.")
except ImportError:
    print("matplotlib no disponible. Instala con: uv add matplotlib")
    print(f"Imágenes preparadas para el LLM: {[p['filename'] for p in photos]}")

---
## 4. Construcción del prompt de sistema

El prompt que recibe GPT-4o no es fijo: se construye dinámicamente inyectando
tres tipos de contexto procedente de la base de datos:

| Tipo de contexto | Origen | Uso en el prompt |
|---|---|---|
| **Correcciones de operadores** | Tabla `feedback_signals` | Muestra pares (IA propuso → operador corrigió) para que el modelo aprenda a evitar esos errores |
| **Historial interno** | Tabla `historical_references` | Precios reales de ventas anteriores en nuestra plataforma |
| **Precios de mercado** | `ExternalComparableClient` (DuckDuckGo) | Media de precios web en tiempo real |

Este mecanismo permite que el modelo **mejore progresivamente** sin necesidad de
fine-tuning: simplemente leyendo el contexto del prompt en cada llamada.

In [ ]:
# Simulamos el contexto de enriquecimiento que el worker extrae de la base de datos
context = {
    "feedback_signals": [
        {
            "field_name": "suggested_price",
            "original_value": "45.00",
            "corrected_value": "35.00",
        },
        {
            "field_name": "description_text",
            "original_value": "Artículo en buen estado.",
            "corrected_value": "Chaqueta de cuero marrón talla M, ligeras marcas de uso en los codos.",
        },
    ],
    "historical_refs": [
        {"sold_price": 38.0, "condition_label": "buen estado", "similarity_score": 0.82},
        {"sold_price": 42.5, "condition_label": "como nuevo", "similarity_score": 0.71},
        {"sold_price": 29.0, "condition_label": "aceptable", "similarity_score": 0.65},
    ],
    # web_prices se añade después de la búsqueda de comparables
}

# Reproducimos la función _build_system_prompt del LLMClient
def build_system_prompt(context: dict) -> str:
    lines = [
        "Eres un experto en valoración de productos de segunda mano para una plataforma de reventa en España.",
        "Analiza las imágenes del producto y genera una descripción atractiva en español y un precio de venta sugerido en EUR.",
        "",
        "Responde ÚNICAMENTE con JSON válido (sin bloques markdown) con exactamente estos campos:",
        '{"description": "...", "suggested_price": 0.00, "confidence": 0.85, "product_keywords": ["..."]}',
        "",
        "Instrucciones:",
        "- description: 2-3 frases en español, describen el producto, su estado y atractivo para el comprador",
        "- suggested_price: precio realista de venta en EUR como número decimal",
        "- confidence: tu nivel de confianza en la valoración entre 0.0 y 1.0",
        "- product_keywords: 3-5 palabras clave en español para buscar comparables de precio en internet",
    ]

    feedback = context.get("feedback_signals", [])
    if feedback:
        lines.append("")
        lines.append("## CORRECCIONES DE OPERADORES HUMANOS (aprende de estos patrones):")
        for f in feedback[-15:]:
            field_label = "descripción" if f["field_name"] == "description_text" else "precio"
            orig = str(f["original_value"])[:100]
            corr = str(f["corrected_value"])[:100]
            lines.append(f"  - {field_label}: IA propuso '{orig}' → operador corrigió a '{corr}'")
        lines.append("  → Ajusta tus respuestas para evitar estos tipos de errores.")

    hist = context.get("historical_refs", [])
    if hist:
        lines.append("")
        lines.append("## VENTAS HISTÓRICAS EN NUESTRA PLATAFORMA (referencia de precios reales):")
        for h in hist[:8]:
            cond = h.get("condition_label") or "sin clasificar"
            sim = h.get("similarity_score")
            sim_txt = f", similitud {sim:.0%}" if sim else ""
            lines.append(f"  - Vendido a {h['sold_price']:.2f}€ (condición: {cond}{sim_txt})")

    web = context.get("web_prices", [])
    if web:
        valid = [w for w in web if w.get("price")]
        if valid:
            avg = sum(w["price"] for w in valid) / len(valid)
            lines.append("")
            lines.append(f"## PRECIOS EN MERCADO ONLINE (media: {avg:.2f}€):")
            for w in valid[:6]:
                title = w.get("title", "")[:60]
                lines.append(f"  - {w['price']:.2f}€ — {title}")

    return "\n".join(lines)


system_prompt = build_system_prompt(context)
print("=" * 70)
print("PROMPT DE SISTEMA GENERADO DINÁMICAMENTE")
print("=" * 70)
print(system_prompt)

---
## 5. Evaluación de imágenes con el modelo de IA (modo stub)

En producción, `LLMClient.propose()` envía las imágenes a GPT-4o y recibe JSON.
En modo stub, devuelve datos sintéticos reproducibles sin coste.

Ejecutamos el `PricingService` completo: LLM → búsqueda web → blending de precios.

In [ ]:
import asyncio

# Intentamos usar el PricingService real del repositorio
try:
    from backend.src.services.pricing_service import PricingService

    service = PricingService()
    proposal = asyncio.run(
        service.build_proposal(photos, query="chaqueta cuero segunda mano", context=context)
    )
    print("PricingService importado desde el repositorio.")

except Exception as exc:
    # Fallback: reproducimos la lógica de PricingService directamente en el notebook
    print(f"Importación directa no disponible ({exc}). Reproduciendo lógica localmente.\n")

    @dataclass
    class LLMResult:
        description: str
        suggested_price: float
        confidence: float
        product_keywords: list = field(default_factory=list)

    def llm_stub() -> LLMResult:
        DESCRIPTIONS = [
            "Chaqueta de cuero marrón talla M en buen estado general, con pequeñas marcas de uso en los codos. Ideal para segunda mano.",
            "Artículo bien conservado, funcionando correctamente. Oportunidad de compra a buen precio.",
        ]
        return LLMResult(
            description=random.choice(DESCRIPTIONS),
            suggested_price=round(random.uniform(30, 55), 2),
            confidence=round(random.uniform(0.55, 0.80), 2),
            product_keywords=["chaqueta cuero", "segunda mano", "talla M"],
        )

    def external_comparable_stub(query: str) -> list:
        return [
            {"price": 39.99, "title": "Chaqueta cuero marrón M — Wallapop"},
            {"price": 45.00, "title": "Cazadora cuero marrón talla M — Vinted"},
            {"price": 33.50, "title": "Chaqueta cuero vintage — eBay España"},
        ]

    llm = llm_stub()
    web_comps = external_comparable_stub(" ".join(llm.product_keywords))
    market_prices = [c["price"] for c in web_comps]
    hist_prices = [h["sold_price"] for h in context.get("historical_refs", [])]
    all_external = market_prices + hist_prices
    avg_external = sum(all_external) / len(all_external) if all_external else llm.suggested_price
    suggested = round(llm.suggested_price * 0.6 + avg_external * 0.4, 2)

    proposal = {
        "description": llm.description,
        "suggested_price": suggested,
        "suggested_price_min": round(suggested * 0.85, 2),
        "suggested_price_max": round(suggested * 1.15, 2),
        "confidence_score": llm.confidence,
        "rationale_internal": {
            "llm_estimate": llm.suggested_price,
            "product_keywords": llm.product_keywords,
            "historical_refs_used": len(hist_prices),
        },
        "rationale_external": {
            "avg_market_price": round(avg_external, 2),
            "web_sources_found": len(market_prices),
            "sources": web_comps,
        },
    }

print("\n" + "=" * 70)
print("PROPUESTA GENERADA POR EL PIPELINE DE IA")
print("=" * 70)
print(json.dumps(proposal, indent=2, ensure_ascii=False))

---
## 6. Interpretación de resultados y métricas

Analizamos la propuesta generada:
- **Trazabilidad interna**: cuánto pesa la estimación del LLM y cuántas referencias históricas se usaron.
- **Trazabilidad externa**: precio medio de mercado y fuentes consultadas.
- **Banda de precio**: rango `[min, max]` con ±15% del precio sugerido.
- **Confianza**: el LLM devuelve un `confidence_score` entre 0 y 1.

In [ ]:
ri = proposal["rationale_internal"]
re_ = proposal["rationale_external"]

print("━" * 60)
print("  DESCRIPCIÓN GENERADA")
print("━" * 60)
print(f"  {proposal['description']}\n")

print("━" * 60)
print("  PRECIO SUGERIDO")
print("━" * 60)
print(f"  Mínimo:    {proposal['suggested_price_min']:.2f} €")
print(f"  Sugerido:  {proposal['suggested_price']:.2f} €  ◄─ precio recomendado")
print(f"  Máximo:    {proposal['suggested_price_max']:.2f} €")
print(f"  Confianza: {proposal['confidence_score']:.0%}\n")

print("━" * 60)
print("  TRAZABILIDAD — SEÑALES INTERNAS")
print("━" * 60)
print(f"  Estimación LLM:            {ri['llm_estimate']:.2f} €  (peso 60%)")
print(f"  Palabras clave producto:   {ri['product_keywords']}")
print(f"  Refs. históricas usadas:   {ri['historical_refs_used']}\n")

print("━" * 60)
print("  TRAZABILIDAD — SEÑALES EXTERNAS (mercado online)")
print("━" * 60)
print(f"  Media de mercado:          {re_['avg_market_price']:.2f} €  (peso 40%)")
print(f"  Fuentes encontradas:       {re_['web_sources_found']}")
for src in re_.get("sources", []):
    price_str = f"{src['price']:.2f} €" if src.get("price") else "—"
    print(f"    · {price_str}  {src.get('title', '')}")

print("\n━" * 60)
# Fórmula de blending
llm_est = ri["llm_estimate"]
mkt_avg = re_["avg_market_price"]
blended = round(llm_est * 0.6 + mkt_avg * 0.4, 2)
print(f"\n  Fórmula: {llm_est:.2f} × 0.6  +  {mkt_avg:.2f} × 0.4  =  {blended:.2f} €")

---
## 7. Visualización de resultados y predicciones

Representamos gráficamente la composición del precio sugerido
(señal LLM vs señales de mercado) y el histograma de precios de referencia.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    ri = proposal["rationale_internal"]
    re_ = proposal["rationale_external"]
    suggested = proposal["suggested_price"]
    p_min = proposal["suggested_price_min"]
    p_max = proposal["suggested_price_max"]

    hist_prices = [h["sold_price"] for h in context["historical_refs"]]
    web_prices = [s["price"] for s in re_.get("sources", []) if s.get("price")]
    all_prices = hist_prices + web_prices

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Gráfico 1: Composición del precio (barras apiladas) ──────────────────
    ax1 = axes[0]
    llm_contrib = ri["llm_estimate"] * 0.6
    mkt_contrib = re_["avg_market_price"] * 0.4
    ax1.bar(["Precio sugerido"], [llm_contrib], color="#4A7CC2", label=f"LLM × 0.6  ({ri['llm_estimate']:.2f} €)")
    ax1.bar(["Precio sugerido"], [mkt_contrib], bottom=[llm_contrib], color="#F5A623", label=f"Mercado × 0.4  ({re_['avg_market_price']:.2f} €)")
    ax1.axhline(suggested, color="black", linestyle="--", linewidth=1.5, label=f"Precio final: {suggested:.2f} €")
    ax1.fill_between([-0.4, 0.4], p_min, p_max, alpha=0.15, color="green", label=f"Banda [{p_min:.2f}–{p_max:.2f} €]")
    ax1.set_ylabel("EUR")
    ax1.set_title("Composición del precio sugerido")
    ax1.legend(fontsize=8)
    ax1.set_ylim(0, p_max * 1.2)

    # ── Gráfico 2: Distribución de precios de referencia ──────────────────────
    ax2 = axes[1]
    if all_prices:
        ax2.hist(all_prices, bins=max(3, len(all_prices)), color="#4A7CC2", edgecolor="white", alpha=0.8)
        ax2.axvline(suggested, color="red", linestyle="--", linewidth=2, label=f"Precio sugerido: {suggested:.2f} €")
        ax2.axvline(re_["avg_market_price"], color="#F5A623", linestyle=":", linewidth=2,
                    label=f"Media mercado: {re_['avg_market_price']:.2f} €")
        ax2.set_xlabel("EUR")
        ax2.set_ylabel("Frecuencia")
        ax2.set_title("Distribución de precios de referencia\n(historial interno + mercado online)")
        ax2.legend(fontsize=8)
    else:
        ax2.text(0.5, 0.5, "Sin datos de referencia disponibles",
                 ha="center", va="center", transform=ax2.transAxes)
        ax2.set_title("Distribución de precios de referencia")

    plt.suptitle(
        f"Pipeline IA — Confianza: {proposal['confidence_score']:.0%}",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib no disponible — instala con: uv add matplotlib")
    print(f"Precio sugerido: {proposal['suggested_price']:.2f} €")
    print(f"Banda: [{proposal['suggested_price_min']:.2f} – {proposal['suggested_price_max']:.2f} €]")

---
## 8. Control de presupuesto diario y proyección de costes

El sistema incluye un guardrail (`CostGuardrailService`) que bloquea las llamadas
al LLM cuando el gasto acumulado del día supera `LLM_DAILY_BUDGET_USD`.

Cada llamada se registra en la tabla `llm_metrics` con coste, latencia y outcome.
El endpoint `GET /api/v1/metrics/llm` devuelve el resumen agregado.

Proyectamos el gasto diario según diferentes volúmenes de productos.

In [ ]:
# Guardrail: comprobación de presupuesto (lógica de CostGuardrailService)
daily_budget_usd = float(os.environ.get("LLM_DAILY_BUDGET_USD", "25"))

# Coste estimado por producto (en producción, leer de GET /api/v1/metrics/llm)
# gpt-4o: ~$0.005 por imagen en modo 'low' + ~$0.002 tokens de texto ≈ $0.02–0.05/producto
coste_estimado_usd = 0.035  # estimación conservadora

scenarios = {
    "Piloto   (50 prod/día)": 50,
    "Normal  (150 prod/día)": 150,
    "Pico    (300 prod/día)": 300,
    "Máximo  (500 prod/día)": 500,
}

print(f"Presupuesto diario configurado: {daily_budget_usd:.2f} USD")
print(f"Coste estimado por producto:    {coste_estimado_usd:.4f} USD\n")
print(f"{'Escenario':<30} {'Coste/día (USD)':>16} {'Dentro presupuesto':>20}")
print("-" * 70)
for name, vol in scenarios.items():
    coste_dia = vol * coste_estimado_usd
    estado = "✓  OK" if coste_dia <= daily_budget_usd else "✗  EXCEDE — guardrail activa"
    print(f"{name:<30} {coste_dia:>16.2f} {estado:>20}")

print()

# Demostración del guardrail
class CostGuardrailDemo:
    def __init__(self, budget: float):
        self.budget = budget

    def check_budget(self, current_cost: float) -> dict:
        if current_cost >= self.budget:
            return {"allowed": False, "reason": "daily_budget_exceeded"}
        return {"allowed": True, "reason": None}

guardrail = CostGuardrailDemo(daily_budget_usd)
test_costs = [0.0, 10.5, 24.9, 25.0, 30.0]
print("Simulación del guardrail con diferentes costes acumulados:")
for cost in test_costs:
    result = guardrail.check_budget(cost)
    symbol = "✓" if result["allowed"] else "✗"
    print(f"  {symbol} Gasto acumulado {cost:5.1f} USD → {result}")

---
## 9. Bucle de retroalimentación — cómo el operador mejora el modelo

Cuando el operador corrige una propuesta en la interfaz de carrusel, el sistema
almacena esa corrección como `FeedbackSignal`. En el siguiente análisis, esas
correcciones aparecen en el prompt del LLM para que el modelo aprenda a evitar
ese tipo de error.

Este es el bucle completo sin fine-tuning:

```
Fotos  →  LLMClient (prompt + contexto)  →  Propuesta
                ↑                                 ↓
          FeedbackSignal             Operador revisa en carrusel
                ↑                                 ↓
          Corrección almacenada  ←  Operador corrige descripción/precio
```

In [ ]:
# Simulamos cómo una corrección del operador afecta al prompt del siguiente análisis

correction = {
    "field_name": "suggested_price",
    "original_value": str(proposal["suggested_price"]),
    "corrected_value": str(round(proposal["suggested_price"] * 0.90, 2)),  # operador baja 10%
}

# Añadimos la corrección al contexto
updated_context = {
    **context,
    "feedback_signals": context["feedback_signals"] + [correction],
}

updated_prompt = build_system_prompt(updated_context)

print("PROMPT ACTUALIZADO tras la corrección del operador:")
print("=" * 70)
# Mostrar solo el bloque de correcciones para claridad
start = updated_prompt.find("## CORRECCIONES")
end = updated_prompt.find("## VENTAS", start)
print(updated_prompt[start:end] if end > 0 else updated_prompt[start:])
print()
print(f"La siguiente llamada al LLM incluirá {len(updated_context['feedback_signals'])} correcciones de operadores.")
print("El modelo ajustará sus estimaciones de precio hacia abajo automáticamente.")

---
## 10. Resumen y próximos pasos

| Componente | Estado | Descripción |
|---|---|---|
| `LLMClient` | ✓ Implementado | GPT-4o vision con prompt dinámico y modo stub |
| `ExternalComparableClient` | ✓ Implementado | DuckDuckGo Search en tiempo real |
| `PricingService` | ✓ Implementado | Blending LLM×0.6 + mercado×0.4 con trazabilidad |
| `CostGuardrailService` | ✓ Implementado | Bloqueo automático al superar presupuesto diario |
| `LLMMetricsService` | ✓ Implementado | Registro y resumen de métricas por llamada |
| Retroalimentación operador | ✓ Implementado | `FeedbackSignal` → prompt del siguiente análisis |
| Interfaz de carrusel | ✓ Implementado | Swipe + botones, undo 5 s, bloqueo optimista |
| Fine-tuning / reentrenamiento | ⏳ Pendiente | Fase posterior con dataset de correcciones acumuladas |
| Publicación externa automática | ⏳ Pendiente | Diferida a v2 según spec feature 001 |

### Cómo usar este notebook con datos reales

1. Configura `OPENAI_API_KEY` en `.env.local` y establece `LLM_STUB=false`.
2. Sube fotos reales de un producto vía `POST /api/v1/images/{product_id}`.
3. Arranca el worker Celery: `uv run celery -A backend.src.workers.celery_app worker`.
4. Ejecuta el análisis desde la API: `POST /api/v1/proposals/{product_id}/generate`.
5. Consulta los resultados en `GET /api/v1/proposals/{product_id}` y en `GET /api/v1/metrics/llm`.
6. Rellena `notebooks/pricing-eval.ipynb` con los datos reales para comparar baseline vs IA.

> Para la evaluación de calidad completa (MAE de precio, tasa de aprobación sin edición, etc.)
> consulta el notebook `pricing-eval.ipynb`.